# scGPT MAPT Tangle Stage — Preprocessing & EDA

**Project:** Fine-tuning scGPT to predict neurofibrillary tangle stage from single-nucleus RNA-seq data
**Dataset:** Morabito et al. 2021 (GSE174367) — Single-nucleus transcriptomic characterization of Alzheimer's disease
**Output:** `neurons_hvg_preprocessed.h5ad` — preprocessed neuron subset ready for scGPT fine-tuning

---
**Pipeline overview:**
1. Load raw 10x expression matrix and cell metadata (**including donor ID**)
2. Explore dataset composition, target distribution and **donor-level confounding**
3. Subset to neurons (biological decision)
4. Quality control, normalization, HVG selection (**raw counts kept in a layer**)
5. MAPT expression across tangle stages — **normalized, and pseudobulk per donor**
6. Baseline UMAP (scaling applied to a **copy** so the saved data stays log1p)
7. Save preprocessed data

**Changes from v1 (see README "Changes"):** donor ID is kept for donor-level splits; `sc.pp.scale` no longer overwrites the saved expression matrix; MAPT analysis uses normalized data and donor-level pseudobulk instead of raw per-cell counts.

## 1. Setup and Imports

**Environment:** `scgpt_mapt` conda environment on Northeastern Discovery HPC. Set `SCGPT_PROJECT_DIR` to override the project path.

In [ ]:
import os
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for HPC
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from importlib.metadata import version

PROJECT_DIR = os.environ.get('SCGPT_PROJECT_DIR', '/scratch/arumuganainar.d/projects/scgpt')
os.chdir(PROJECT_DIR)
os.makedirs('figures', exist_ok=True)

import sys
sys.path.insert(0, PROJECT_DIR)
from splits import find_donor_column

sc.settings.verbosity = 1
print('Scanpy version:', version('scanpy'))
print('Working directory:', os.getcwd())

## 2. Data Loading

**Files (GEO GSE174367):**
- `GSE174367_snRNA-seq_filtered_feature_bc_matrix.h5` — 10x expression matrix
- `GSE174367_snRNA-seq_cell_meta.csv.gz` — cell metadata (tangle stage, diagnosis, cell type, **sample/donor ID**, PMI, RIN)

The donor ID is essential: tangle stage is a property of the donor, so all later train/test splits are made by donor.

In [ ]:
def load_dataset(h5_path, meta_path):
    """
    Load 10x expression matrix and attach cell metadata, including donor ID.

    Returns
    -------
    adata : AnnData with metadata in obs (donor ID stored as 'donor_id')
    meta_aligned : pd.DataFrame
    """
    adata = sc.read_10x_h5(h5_path)
    adata.var_names_make_unique()

    meta = pd.read_csv(meta_path, index_col=0)
    print(f'Metadata columns: {meta.columns.tolist()}')

    common_barcodes = adata.obs_names.intersection(meta.index)
    adata = adata[common_barcodes].copy()
    meta_aligned = meta.loc[common_barcodes]

    donor_col = find_donor_column(meta_aligned.columns)
    print(f'Using "{donor_col}" as the donor ID column')
    adata.obs['donor_id'] = meta_aligned[donor_col].astype(str).values

    keep = ['Diagnosis', 'Cell.Type', 'Tangle.Stage', 'Age', 'Sex', 'Batch', 'PMI', 'RIN']
    for col in keep:
        if col in meta_aligned.columns:
            adata.obs[col] = meta_aligned[col].values

    return adata, meta_aligned


adata, meta = load_dataset(
    'GSE174367_snRNA-seq_filtered_feature_bc_matrix.h5',
    'GSE174367_snRNA-seq_cell_meta.csv.gz'
)
print(f'Loaded: {adata.n_obs:,} cells x {adata.n_vars:,} genes from {adata.obs["donor_id"].nunique()} donors')

## 3. Dataset Exploration and Donor-Level Confounding

Because every cell of a donor shares one tangle stage, the **effective sample size is the number of donors**, not the number of cells. We check:
- how many donors each stage has (this limits how many cross-validation folds are possible)
- whether stage is confounded with diagnosis or sequencing batch at the donor level

In [ ]:
print('=== Cell Type Distribution ===')
print(adata.obs['Cell.Type'].value_counts())

print('\n=== Tangle Stage Distribution (cells) ===')
print(adata.obs['Tangle.Stage'].value_counts().sort_index())

donor_info = (adata.obs
              .groupby('donor_id', observed=True)
              .agg(stage=('Tangle.Stage', 'first'),
                   diagnosis=('Diagnosis', 'first'),
                   batch=('Batch', 'first'),
                   n_cells=('Tangle.Stage', 'size')))
print(f'\n=== Donors: {len(donor_info)} ===')
print(donor_info.sort_values('stage').to_string())

print('\n=== Donors per stage ===')
print(donor_info['stage'].value_counts().sort_index())

print('\n=== Stage x Diagnosis (donors) ===')
print(pd.crosstab(donor_info['stage'], donor_info['diagnosis']))

print('\n=== Stage x Batch (donors) ===')
print(pd.crosstab(donor_info['stage'], donor_info['batch']))

missing_cols = [c for c in ['Tangle.Stage', 'Cell.Type', 'Diagnosis', 'PMI', 'RIN'] if c in adata.obs]
print('\n=== Missing Values ===')
print(adata.obs[missing_cols].isnull().sum())

**Interpretation guide (fill in after running):**
- Stages with very few donors make any per-stage conclusion fragile.
- If Stage 1–2 donors are all Controls and Stage 5–6 all AD, the task is partly AD-vs-Control.
- If stage and batch overlap heavily, a model can learn batch instead of pathology — donor-level evaluation plus baselines are the guard against this.

## 4. Neuron Subsetting

Neurofibrillary tau tangles form **primarily in neurons**. We keep excitatory (EX) and inhibitory (INH) neurons and drop glia (ODC, ASC, MG, OPC, PER.END).

In [ ]:
neurons = adata[adata.obs['Cell.Type'].isin(['EX', 'INH'])].copy()

print(f'Total cells: {adata.n_obs:,}')
print(f'Neurons only: {neurons.n_obs:,} ({neurons.n_obs/adata.n_obs*100:.1f}% of total)')
print(neurons.obs['Cell.Type'].value_counts())
print(f'\nNeurons per donor (min / median / max): '
      f'{neurons.obs["donor_id"].value_counts().min()} / '
      f'{int(neurons.obs["donor_id"].value_counts().median())} / '
      f'{neurons.obs["donor_id"].value_counts().max()}')
print(f'MAPT present: {"MAPT" in neurons.var_names}')

## 5. Quality Control and Preprocessing

1. **QC** — min 200 genes per cell, min 10 cells per gene
2. **Keep raw counts** in `layers['counts']` (needed for pseudobulk and for re-processing)
3. **Normalize** to 10,000 counts per cell + **log1p** — this log1p matrix stays in `X` and is what scGPT is tokenized from (scGPT's own binning is applied in the fine-tuning notebook)
4. **HVG selection** — 3,000 genes, MAPT force-included (it is stably expressed, so not "variable", but it is the gene of interest)
5. **Label encoding** — Stage 1/2/5/6 → 0/1/2/3 (Stages 3 and 4 are absent)

In [ ]:
def preprocess_neurons(neurons, n_hvg=3000):
    """QC, normalization and HVG selection. Returns (full normalized neurons, HVG subset)."""
    sc.pp.filter_cells(neurons, min_genes=200)
    sc.pp.filter_genes(neurons, min_cells=10)
    print(f'After QC: {neurons.n_obs:,} cells x {neurons.n_vars:,} genes')

    neurons.layers['counts'] = neurons.X.copy()          # raw counts kept

    sc.pp.normalize_total(neurons, target_sum=1e4)
    sc.pp.log1p(neurons)

    sc.pp.highly_variable_genes(neurons, n_top_genes=n_hvg)
    neurons.var.loc['MAPT', 'highly_variable'] = True
    print(f'HVGs (incl. forced MAPT): {neurons.var.highly_variable.sum()}')

    neurons_hvg = neurons[:, neurons.var.highly_variable].copy()
    print(f'Final matrix: {neurons_hvg.n_obs:,} cells x {neurons_hvg.n_vars:,} genes')

    stage_mapping = {'Stage 1': 0, 'Stage 2': 1, 'Stage 5': 2, 'Stage 6': 3}
    neurons_hvg.obs['tangle_label'] = neurons_hvg.obs['Tangle.Stage'].map(stage_mapping).astype(int)
    print(neurons_hvg.obs['tangle_label'].value_counts().sort_index())
    return neurons, neurons_hvg


neurons, neurons_hvg = preprocess_neurons(neurons)

## 6. MAPT Expression Across Tangle Stages

v1 plotted **raw counts per cell**, which mixes biology with sequencing depth, and treated 12k cells as independent. Here we use:
- **Cell level:** log-normalized MAPT (depth-corrected), for visualization only
- **Donor level (pseudobulk):** sum raw counts per donor → MAPT counts-per-million. Statistical tests use **donors as the unit** (Kruskal–Wallis across stages, Spearman with stage order)

Any stage trend should be read as a **hypothesis** (e.g. survivor bias), given the small number of donors per stage.

In [ ]:
stages = ['Stage 1', 'Stage 2', 'Stage 5', 'Stage 6']

# Cell level, normalized
mapt_norm = np.asarray(neurons[:, 'MAPT'].X.todense()).ravel() if hasattr(neurons.X, 'todense') \
    else np.asarray(neurons[:, 'MAPT'].X).ravel()
neurons.obs['MAPT_log1p'] = mapt_norm

# Donor level pseudobulk CPM
counts = neurons.layers['counts']
mapt_idx = neurons.var_names.get_loc('MAPT')
pb = []
for donor, idx in neurons.obs.groupby('donor_id', observed=True).indices.items():
    sub = counts[idx]
    total = float(sub.sum())
    mapt = float(sub[:, mapt_idx].sum())
    pb.append({'donor_id': donor,
               'stage': neurons.obs['Tangle.Stage'].iloc[idx[0]],
               'n_cells': len(idx),
               'MAPT_CPM': mapt / total * 1e6})
pb = pd.DataFrame(pb).sort_values('stage')
print(pb.to_string(index=False))

groups = [pb.loc[pb.stage == s, 'MAPT_CPM'].values for s in stages if (pb.stage == s).any()]
kw = stats.kruskal(*groups)
stage_num = pb['stage'].map({'Stage 1': 1, 'Stage 2': 2, 'Stage 5': 5, 'Stage 6': 6})
rho = stats.spearmanr(stage_num, pb['MAPT_CPM'])
print(f'\nDonor-level Kruskal-Wallis: H={kw.statistic:.2f}, p={kw.pvalue:.3f}')
print(f'Donor-level Spearman (stage order vs MAPT CPM): rho={rho.correlation:.2f}, p={rho.pvalue:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.violinplot(data=neurons.obs, x='Tangle.Stage', y='MAPT_log1p', order=stages,
               ax=axes[0], inner='quartile', cut=0)
axes[0].set_title('MAPT per cell (log-normalized)')
axes[0].set_ylabel('log1p(normalized MAPT)')
sns.boxplot(data=pb, x='stage', y='MAPT_CPM', order=stages, ax=axes[1], color='lightgrey')
sns.stripplot(data=pb, x='stage', y='MAPT_CPM', order=stages, ax=axes[1], size=7)
axes[1].set_title(f'MAPT pseudobulk per donor (n={len(pb)})\nKruskal p={kw.pvalue:.3f}')
axes[1].set_ylabel('MAPT counts per million')
plt.tight_layout()
plt.savefig('figures/MAPT_by_tangle_stage.png', dpi=150)
pb.to_csv('figures/MAPT_pseudobulk_by_donor.csv', index=False)
print('Saved figures/MAPT_by_tangle_stage.png and figures/MAPT_pseudobulk_by_donor.csv')

## 7. Baseline UMAP

PCA/UMAP needs scaled data, but scaling **must not overwrite** the log1p matrix that scGPT is tokenized from (this was a bug in v1: `sc.pp.scale` ran in place before saving, so scGPT received z-scores). We scale a **copy** and only carry the embeddings back.

UMAP is a visualization, not a classifier — whether stages are predictable is tested with proper baselines (`baselines.py`) on donor-level folds.

In [ ]:
tmp = neurons_hvg.copy()
sc.pp.scale(tmp, max_value=10)
sc.pp.pca(tmp, n_comps=50)
sc.pp.neighbors(tmp, n_neighbors=15, n_pcs=40)
sc.tl.umap(tmp)

neurons_hvg.obsm['X_pca'] = tmp.obsm['X_pca']
neurons_hvg.obsm['X_umap'] = tmp.obsm['X_umap']
neurons_hvg.obs['MAPT_log1p'] = neurons.obs.loc[neurons_hvg.obs_names, 'MAPT_log1p'].values
del tmp

assert neurons_hvg.X.min() >= 0, 'X must stay log1p (non-negative)'

fig, axes = plt.subplots(1, 4, figsize=(24, 5))
sc.pl.umap(neurons_hvg, color='Tangle.Stage', ax=axes[0], show=False, title='Tangle Stage')
sc.pl.umap(neurons_hvg, color='MAPT_log1p', ax=axes[1], show=False, title='MAPT (log1p)', color_map='Reds')
sc.pl.umap(neurons_hvg, color='Cell.Type', ax=axes[2], show=False, title='Neuron Type')
sc.pl.umap(neurons_hvg, color='donor_id', ax=axes[3], show=False, title='Donor', legend_loc=None)
plt.tight_layout()
plt.savefig('figures/UMAP_neurons_baseline.png', dpi=150, bbox_inches='tight')
print('Saved figures/UMAP_neurons_baseline.png')

**What to look for:** if cells cluster by **donor** more than by stage, a cell-level random split would let a model "cheat" by recognizing donors — which is why every model in this project is evaluated on held-out donors.

## 8. Save Preprocessed Data

- `X`: log1p-normalized expression, 3,000 HVGs + MAPT (**not scaled**)
- `layers['counts']`: raw counts for the same genes
- `obs`: Tangle.Stage, tangle_label, **donor_id**, Cell.Type, Diagnosis, Age, Sex, Batch, PMI, RIN
- `obsm`: PCA and UMAP

In [ ]:
out = 'neurons_hvg_preprocessed.h5ad'
neurons_hvg.write_h5ad(out)
print(f'Saved: {out} ({os.path.getsize(out)/1024**2:.1f} MB)')
print(f'  Cells: {neurons_hvg.n_obs:,}  Genes: {neurons_hvg.n_vars:,}  Donors: {neurons_hvg.obs["donor_id"].nunique()}')
print(f'  X range: {neurons_hvg.X.min():.2f} to {neurons_hvg.X.max():.2f} (log1p, should be >= 0)')
print(f'  Layers: {list(neurons_hvg.layers.keys())}')
print('\nNext: python baselines.py, then scgpt_finetune.ipynb')